# Transaction Fraud Intelligence — Frozen Final Evaluation

This notebook is **not another modelling round**. V2 development was frozen before the reserved period was materialized.

It evaluates the frozen protocol on synthetic days **102–119 exactly once** using the preselected `catboost_history` primary model, the same five seeds, the same four model comparison and the same 20/50/100 daily review budgets.

Before scoring anything, the runner verifies the exact frozen V2 engine SHA-256. For every seed it then proves that the entire development transaction prefix and all development point-in-time features are **exactly unchanged**. Any mismatch stops the run.

**Important:** selecting **Runtime → Run all** intentionally materializes and evaluates the reserved synthetic period. After a valid run, it is no longer unseen. Whatever result appears should be reported without post-hoc tuning.


In [ ]:
import hashlib, json, os, platform, subprocess, sys, urllib.request
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import FileLink, display

FROZEN_COMMIT = 'a4ab1de96e2d0bed7fa1f19836f9585f3f10344f'
FINAL_CODE_COMMIT = 'febc1c6b5ae7d87896342948b24ca1fd66f07a13'
FROZEN_ENGINE_SHA256 = '6debf4f7501ee0f937bd53c9992f8ecc642c946039dd653aa284229078d2cb1e'
REPO_RAW = 'https://raw.githubusercontent.com/aaravb015/transaction-fraud-intelligence'
ACK = 'EVALUATE_RESERVED_FINAL_TEST_ONCE'

if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('Use a Colab Python 3.11–3.13 runtime.')

def fetch(url):
    with urllib.request.urlopen(url, timeout=60) as r:
        return r.read()

engine_bytes = fetch(f'{REPO_RAW}/{FROZEN_COMMIT}/src/fraud_v2.py')
actual = hashlib.sha256(engine_bytes).hexdigest()
if actual != FROZEN_ENGINE_SHA256:
    raise RuntimeError('Frozen V2 engine hash mismatch. Final test has NOT been evaluated.')

requirements_bytes = fetch(f'{REPO_RAW}/{FROZEN_COMMIT}/requirements.txt')
requirements_text = requirements_bytes.decode('utf-8')
digest = hashlib.sha256(requirements_bytes).hexdigest()[:12]
runtime = Path.cwd()/'.fraud_final_runtime'/(f'py{sys.version_info.major}{sys.version_info.minor}_'+digest)
packages = runtime/'packages'
runtime.mkdir(parents=True, exist_ok=True)
req = runtime/'requirements.txt'
req.write_text(requirements_text, encoding='utf-8')
marker = runtime/'installed.txt'
if not marker.exists() or marker.read_text(encoding='utf-8') != requirements_text:
    print('Installing frozen experiment dependencies in isolation...', flush=True)
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--only-binary=:all:', '--ignore-installed', '--upgrade', '--target', str(packages), '-r', str(req)], capture_output=True, text=True)
    if result.returncode:
        print(result.stdout); print(result.stderr)
        raise RuntimeError('Dependency installation failed. Final test has NOT been evaluated.')
    marker.write_text(requirements_text, encoding='utf-8')

env = dict(os.environ, PYTHONPATH=str(packages), PYTHONNOUSERSITE='1', MPLBACKEND='Agg')
probe = "import numpy,pandas,scipy,sklearn,catboost,matplotlib; import platform; print('Final-eval Python:',platform.python_version())"
subprocess.run([sys.executable, '-S', '-c', probe], env=env, check=True)

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
WORK = Path.cwd()/'outputs'/('final_work_'+run_id)
WORK.mkdir(parents=True)
FROZEN_ENGINE = WORK/'fraud_v2_frozen.py'
FROZEN_ENGINE.write_bytes(engine_bytes)
for name in ['final_eval_continuation.py', 'final_eval_runner.py']:
    (WORK/name).write_bytes(fetch(f'{REPO_RAW}/{FINAL_CODE_COMMIT}/src/{name}'))
print('Frozen implementation verified:', FROZEN_COMMIT)
print('Final-evaluation code pinned:', FINAL_CODE_COMMIT)
print('Reserved period is still unevaluated. The next cell is the one-time evaluation.')


## One-time evaluation

Running the next cell is the intentional unlock. It performs no feature selection, ablation, seed selection or hyperparameter search. It produces a separate final-results ZIP.


In [ ]:
command = [sys.executable, '-S', '-u', str(WORK/'final_eval_runner.py'), '--frozen-engine', str(FROZEN_ENGINE), '--work-dir', str(WORK), '--customers', '1200', '--iterations', '600', '--threads', '2', '--ack', ACK]
process = subprocess.Popen(command, cwd=str(WORK), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError('Final evaluation failed. Stop and share the error; do not tune the model from partial output.')
result = json.loads((WORK/'final_result.json').read_text(encoding='utf-8'))
FINAL_ZIP = Path(result['archive'])
print('Final results ZIP:', FINAL_ZIP)
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(FINAL_ZIP.relative_to(Path.cwd()))))
else:
    files.download(str(FINAL_ZIP))


## After the run

Upload the downloaded `fraud_final_....zip` for independent review. Do **not** rerun because of disappointing metrics, and do not merge to `main` until the final evidence and README wording have been reviewed.
